# CortexTwin: training three LoRA adapters on one base model

**What this notebook does, in one paragraph:** we take one big pretrained model (Llama-3.1-8B-Instruct) and teach it three separate skills. Each skill is stored in its own small "adapter" file. The big model itself is never changed.

| Adapter | Skill | Reads | Writes |
|---|---|---|---|
| **Prober** | Asks Socratic questions | The conversation so far | `<thought>…</thought>` + one open question |
| **Diagnoser** | Names what the learner got wrong | A finished transcript | `{"topic", "misconception"}` JSON |
| **Graph** | Maps what the learner knows | A finished transcript | Concept graph JSON (nodes + status + prerequisite edges) |

**How they work together later:** the Prober runs the conversation. When it is over, the Diagnoser and the Graph adapter read the transcript and produce the report.

### Mini glossary (read once, then refer back)
- **Token**: a chunk of text, roughly ¾ of a word. Models read and write tokens, not letters.
- **Weights / parameters**: the billions of numbers inside the model. "Training" means nudging these numbers.
- **LoRA adapter**: instead of changing the 8 billion weights, we add tiny extra matrices next to some layers and only train those (~0.5% of the size). The result is a small folder (~160 MB) you can switch on and off.
- **QLoRA**: LoRA where the big frozen model is stored in 4-bit numbers to save GPU memory. That is what lets an 8B model train on Kaggle's free T4 GPUs.
- **Epoch**: one full pass over the training data.
- **Batch**: how many examples the model looks at before it updates its weights once.
- **Loss**: a score of how wrong the model's predictions were. Lower is better. *Train loss* is measured on data it learns from, *eval/validation loss* on data it never learns from.
- **Chat template**: the exact text format (special tokens like `<|start_header_id|>`) that Llama expects around each message.

### Order to run
Run top to bottom. Training cells each take roughly 5–10 minutes on Kaggle's 2× T4. After training, the evaluation section loads all three adapters at once and scores each one against the base model.

## 1. Install the libraries
- **Unsloth**: makes fine-tuning ~2x faster and uses less memory. It wraps the libraries below.
- **transformers** (comes with Unsloth): loads models and tokenizers from Hugging Face.
- **trl**: provides `SFTTrainer`, the training loop for Supervised Fine-Tuning.
- **peft**: implements LoRA adapters.
- **bitsandbytes**: does the 4-bit compression of the model (the "Q" in QLoRA).
- **accelerate**: places the model on the GPUs and splits it across both when there are two.
- **datasets**: loads our `.jsonl` files into a table the trainer understands.
- **xformers**: faster attention math on older GPUs like the T4.

In [ ]:
# The "!" at the start means "run this as a terminal command", not as Python.
# Upgrade pip (Python's package installer) so it can handle new package formats.
!pip install --upgrade pip
# Install Unsloth straight from its GitHub repo. "[kaggle-new]" picks the dependency set that
# matches Kaggle's pre-installed PyTorch, so we don't break it.
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
# Install the helper libraries. "--no-deps" = don't let them pull in their own versions of
# PyTorch etc., which could clash with what Kaggle and Unsloth already set up.
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets

## 2. GPU setup: Kaggle's "GPU T4 x2"
Pick **Accelerator → GPU T4 x2** in the Kaggle settings. The model is split across the two GPUs: roughly the first half of its layers sits on GPU 0 and the rest on GPU 1. This is called *model parallelism*. Each GPU holds less, which leaves more room for long examples. It is **not** *data parallelism* (both GPUs training on different examples at once), so it is not 2× faster. The training log shows this as `Num GPUs used = 2 | Data Parallel GPUs = 1`.

### The hardware: what "GPU T4 x2" gives you
| | Per GPU | Both GPUs |
|---|---|---|
| GPU | NVIDIA Tesla T4 (Turing architecture, 2018) | 2 separate cards |
| GPU memory (VRAM) | 16 GB GDDR6 (~14.6 GB usable, see the loading log) | 32 GB, but **not one shared pool**: each card only sees its own 16 GB |
| CUDA cores | 2,560 | 5,120 |
| Tensor Cores (special units for the matrix maths in neural networks) | 320 | 640 |
| Speed in FP16 (half precision) | ~65 TFLOPS on Tensor Cores | ~130 |
| Power | ~70 W, a small inference-oriented card | |
| Machine around it | | 4 CPU cores, ~29 GB system RAM |

**Why the 32 GB isn't one pool:** a model can't spill across two cards by itself. Something has to decide which layers live where. Here, Unsloth/Accelerate do that automatically: the loading log prints a "budget" per card and puts the final layer (`lm_head`) on `cuda:1`.

**T4 x2 vs P100 (the other free Kaggle option):** the P100 is one card with 16 GB, and it is faster per card at full-precision (FP32) maths. The T4 x2 gives two cards (more total memory) and Tensor Cores for FP16, which is what we use. Both draw from the same weekly free GPU quota.

**Things the T4 can't do, and why the code works around them:**
- **No bfloat16.** bf16 is the number format most modern training uses; it arrived with the next GPU generation (Ampere, e.g. A100). So the trainer uses **fp16** (`fp16 = not is_bfloat16_supported()`). The log confirms it: `Bfloat16 = FALSE`.
- **No FlashAttention-2.** That fast attention kernel needs Ampere or newer. We use **xformers** instead (log: `FA2 = False`, `Xformers = 0.0.35`).

### The memory maths: why this fits at all
GPU memory is the real limit in fine-tuning, so it helps to do the arithmetic. 8B parameters ≈ 8 billion numbers.

| What is in memory | Full fine-tune (fp16 + Adam) | Our QLoRA run |
|---|---|---|
| Model weights | 8B × 2 bytes ≈ **16 GB** | 8B × 0.5 byte (4-bit) ≈ 4 GB, plus the input/output layers kept at higher precision ≈ **5.5 GB**, frozen |
| Trainable weights | all 8B | 42M LoRA weights (0.52%) ≈ 0.08 GB |
| Gradients (how to change each trainable weight) | 8B × 2 bytes ≈ 16 GB | ≈ 0.08 GB |
| Optimizer state (Adam keeps 2 running averages per trainable weight) | 8B × 2 × 4 bytes ≈ **64 GB** | 8-bit Adam on 42M weights ≈ 0.08 GB |
| **Total before activations** | **~100 GB**: impossible on T4s | **~6 GB**: fits easily |

**Activations** are the in-between results of every layer, kept so the backward pass can compute gradients. They grow with batch size × sequence length. That is why `per_device_train_batch_size=2` and `use_gradient_checkpointing="unsloth"` matter: checkpointing throws most activations away and recomputes them when needed, trading extra compute for a lot less memory.

**How your run used the two cards:** the 5.45 GB of 4-bit weights were split about 2.85 GB on GPU 0 and 2.60 GB on GPU 1, leaving ~10 GB per card for activations. That is *model parallelism*: data flows through GPU 0's layers, then GPU 1's. Only one card is busy at any moment, so it is not faster than one card. It just gives more room. The other approach, *data parallelism* (PyTorch DDP: each card holds a full copy and trains on different examples at once), would be ~2× faster, but needs the whole model to fit on each card. Log line: `Num GPUs used = 2 | Data Parallel GPUs = 1` = model split, no data parallelism.

In [ ]:
import torch  # PyTorch, the math engine that runs the model on the GPU

# Check that Kaggle really gave us both GPUs. Expect: 2 GPUs, each 'Tesla T4' with ~15 GB.
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)  # name and memory size of GPU number i
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB")

## 3. Settings in one place
Everything you might want to change lives here: where the data is, which base model, and which adapters to train.

**Before running:** upload `diag_train.jsonl`, `diag_eval.jsonl`, `graph_train.jsonl` and `graph_eval.jsonl` into the same Kaggle dataset as `sft_train.jsonl` and `sft_eval.jsonl`.

In [ ]:
# Folder on Kaggle where your uploaded dataset files live (read-only).
DATA_DIR = "/kaggle/input/datasets/mohamedshafeekt/cortextwin-prober-lora-adapter"

# The base model we fine-tune. This is Llama-3.1-8B-Instruct already compressed to 4-bit by
# Unsloth ("bnb-4bit" = bitsandbytes 4-bit), so it downloads faster and fits in T4 memory.
BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"

# The longest example (in tokens) the model will see. Our longest example is ~700 tokens,
# so 2048 is comfortably enough. Longer = more GPU memory.
MAX_SEQ_LENGTH = 2048

# One entry per adapter: which file to learn from, which file to test on, and which folder
# to save the trained adapter into (saved under /kaggle/working, i.e. the notebook's output).
ADAPTERS = {
    "prober":    {"train": "sft_train.jsonl",   "eval": "sft_eval.jsonl",   "save_to": "socratic_assessor_lora"},
    "diagnoser": {"train": "diag_train.jsonl",  "eval": "diag_eval.jsonl",  "save_to": "diagnoser_lora"},
    "graph":     {"train": "graph_train.jsonl", "eval": "graph_eval.jsonl", "save_to": "graph_lora"},
}

## 4. The training recipe (one function, used three times)
All three adapters are trained the exact same way. Only the data changes. So instead of copying the same code three times, we put it in one function, `train_adapter(name)`.

What happens inside, step by step:
1. **Load the base model fresh** in 4-bit. Fresh every time, so the adapters never mix.
2. **Apply the Llama-3.1 chat template**: turn each list of messages into one long string with Llama's special tokens.
3. **Load the train and eval files** and convert each conversation into that string.
4. **Attach empty LoRA adapters** to the model. Only these will learn.
5. **Build the trainer** with the training settings (batch size, learning rate, epochs…).
6. **Mask the loss**: tell the trainer to learn only from the assistant's replies, not from the system prompt or the user's messages.
7. **Train**, print the loss per epoch, and **save** the adapter folder.
8. **Free the GPU memory**, so the next adapter can load a fresh model.

In [ ]:
import gc     # "garbage collector": lets us force Python to free memory we no longer use
import torch  # PyTorch, the math engine that runs the model on the GPU

# Unsloth must be imported before trl/transformers so it can patch them to run faster.
from unsloth import FastLanguageModel, is_bfloat16_supported
# get_chat_template: gives the tokenizer Llama-3.1's message format.
# train_on_responses_only: hides everything except the assistant's replies from the loss.
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset          # reads our .jsonl files
from trl import SFTTrainer, SFTConfig      # the training loop and its settings
from kaggle_secrets import UserSecretsClient  # reads secrets you stored in Kaggle

# Your Hugging Face token, stored in Kaggle under Add-ons > Secrets as "HF_TOKEN".
# It proves to Hugging Face that you are allowed to download Llama.
hf_token = UserSecretsClient().get_secret("HF_TOKEN")


def train_adapter(name, epochs=3):
    cfg = ADAPTERS[name]  # look up this adapter's file names and save folder

    # ---- Step 1: load the base model, fresh ----------------------------------------------
    # Returns two things: the model (the neural network) and the tokenizer (text <-> tokens).
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,          # which model to download
        max_seq_length=MAX_SEQ_LENGTH,  # longest input we will feed it
        dtype=None,                     # number format; None = auto (float16 on a T4)
        load_in_4bit=True,              # compress weights to 4-bit: ~16 GB -> ~5.5 GB. This makes it QLoRA.
        token=hf_token,                 # our Hugging Face login
    )

    # ---- Step 2: chat template -----------------------------------------------------------
    # Tells the tokenizer to format conversations the way Llama-3.1 was trained to read them:
    # <|start_header_id|>user<|end_header_id|> ...message... <|eot_id|>  (eot = "end of turn")
    tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

    # A small helper that turns a batch of conversations into a batch of plain strings.
    def to_text(batch):
        # batch["messages"] is a list of conversations; each conversation is a list of
        # {"role": ..., "content": ...} dicts. apply_chat_template glues one conversation
        # into a single string. tokenize=False = give back text, not token numbers.
        # add_generation_prompt=False = the conversation is complete; don't add an empty
        # "assistant:" header at the end (that is only needed when asking for a new reply).
        return {"text": [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
                         for c in batch["messages"]]}

    # ---- Step 3: load the data -----------------------------------------------------------
    # load_dataset("json", ...) reads a .jsonl file (one JSON object per line) into a table.
    # split="train" is just the default name the library gives a single file; it does NOT
    # mean this is training data. .map(to_text, batched=True) adds a "text" column to every row.
    train_ds = load_dataset("json", data_files=f"{DATA_DIR}/{cfg['train']}", split="train").map(to_text, batched=True)
    eval_ds = load_dataset("json", data_files=f"{DATA_DIR}/{cfg['eval']}", split="train").map(to_text, batched=True)
    print(f"[{name}] {len(train_ds)} training examples, {len(eval_ds)} evaluation examples")

    # ---- Step 4: attach LoRA adapters ----------------------------------------------------
    # The big model is now frozen. get_peft_model adds small trainable matrices next to the
    # layers listed in target_modules. Only those small matrices will change during training.
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,  # "rank": how big the adapter matrices are. Bigger = can learn more, but slower and easier to overfit.
        # Which layers get an adapter. q/k/v/o are the attention layers (how words look at each other);
        # gate/up/down are the feed-forward layers (where most "knowledge" processing happens). All 7 = standard choice.
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,      # how strongly the adapter's output is mixed in (alpha / r = 1.0 here)
        lora_dropout=0,     # randomly switching off parts of the adapter while training; 0 = off (Unsloth's fast path)
        bias="none",        # don't train the small "bias" numbers; standard for LoRA
        use_gradient_checkpointing="unsloth",  # saves GPU memory by recomputing some values instead of storing them
        random_state=3407,  # fixed random seed, so the starting adapter is the same every run
    )

    # ---- Step 5: the trainer -------------------------------------------------------------
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,  # examples it learns from
        eval_dataset=eval_ds,    # examples it is only tested on (held-out topics)
        args=SFTConfig(
            dataset_text_field="text",       # which column holds the formatted string
            max_seq_length=MAX_SEQ_LENGTH,   # cut off anything longer than this
            dataset_num_proc=2,              # use 2 CPU processes to tokenize the data
            packing=False,                   # don't glue several short examples into one long row
            per_device_train_batch_size=2,   # examples processed at once on the GPU
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=4,   # add up 4 small batches before updating -> acts like batch size 2 x 4 = 8
            warmup_steps=5,                  # start with a tiny learning rate and ramp up over 5 updates (avoids early shocks)
            num_train_epochs=epochs,         # how many times to go through all the training data
            learning_rate=2e-4,              # how big each weight update is (0.0002). Typical for LoRA.
            fp16=not is_bfloat16_supported(),  # T4 can't do bfloat16, so use float16 numbers
            bf16=is_bfloat16_supported(),      # newer GPUs (A100, L4...) would use bfloat16 instead
            logging_steps=5,                 # print the train loss every 5 updates
            eval_strategy="epoch",           # measure eval loss at the end of every epoch
            optim="adamw_8bit",              # the optimizer (the rule for updating weights), stored in 8-bit to save memory
            weight_decay=0.01,               # gently pulls weights toward zero; a mild guard against overfitting
            lr_scheduler_type="linear",      # learning rate goes down in a straight line until the end
            seed=3407,                       # fixed seed for the data shuffling, so runs are repeatable
            output_dir=f"outputs/{name}",    # where temporary checkpoints go
            report_to="none",                # don't send logs to Weights & Biases or similar
        ),
    )
    # (You may see "We found double BOS tokens - we shall remove one". BOS = "beginning of
    # sequence" token. The chat template already adds one and the tokenizer adds another;
    # Unsloth removes the duplicate. Nothing to fix.)

    # ---- Step 6: learn only from the assistant's replies --------------------------------
    # Without this, the model would also be trained to predict the system prompt and the
    # learner's messages, which is wasted effort and can teach it to talk like the learner.
    # These two strings are the exact headers Llama uses before a user turn and an assistant
    # turn. Everything from a user header up to the next assistant header gets label -100,
    # which means "ignore this token when computing the loss".
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
    )
    # Sanity check: decode only the tokens that are NOT -100, i.e. what the model will be
    # graded on. You should see only assistant replies here, no system prompt, no learner text.
    graded = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
    print(f"[{name}] what the model is trained to produce (first example, first 400 chars):")
    print(tokenizer.decode(graded)[:400], "\n")

    # ---- Step 7: train and save ----------------------------------------------------------
    trainer.train()  # the actual learning. A progress table appears while it runs.

    # Print a simple loss table. trainer.state.log_history is a list of everything that got
    # logged; entries with "loss" are training loss, entries with "eval_loss" are eval loss.
    for row in trainer.state.log_history:
        if "eval_loss" in row:
            print(f"[{name}] epoch {row['epoch']:.0f}: eval loss {row['eval_loss']:.3f}")
    # Healthy pattern: eval loss goes down, then flattens. If it goes UP while train loss keeps
    # falling, the model is memorising the training data (overfitting): use fewer epochs.

    # save_pretrained writes ONLY the adapter (small), not the 8B base model.
    model.save_pretrained(cfg["save_to"])
    tokenizer.save_pretrained(cfg["save_to"])  # also save the tokenizer + chat template with it
    print(f"[{name}] adapter saved to {cfg['save_to']}/")

    # ---- Step 8: free the GPU -------------------------------------------------------------
    # 'del' removes our references to the model and trainer; gc.collect() actually frees them;
    # empty_cache() hands the GPU memory back so the next adapter can load a fresh model.
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

## 5. Train adapter 1: the Prober
Learns to reply with `<thought>…</thought>` followed by exactly one open question.

Expected: 80 examples × 3 epochs ÷ batch 8 = **30 training steps**. Last run: eval loss 2.140 → 1.921 → 1.904.

In [ ]:
train_adapter("prober")  # run the whole recipe above with the Prober's data

## 6. Train adapter 2: the Diagnoser
Learns to read a finished transcript and output `{"topic": ..., "misconception": ...}`.

The answer is a short JSON line, so there are far fewer graded tokens per example than for the Prober. **Expect the loss to start lower.** That is because the task is shorter, not because it is "better".

In [ ]:
train_adapter("diagnoser")  # same recipe, Diagnoser data

## 7. Train adapter 3: the Graph
Learns to read a finished transcript and output a concept graph: 5 concepts, each marked `known`, `shaky`, `missing` or `untested`, plus which concept is a prerequisite of which.

In [ ]:
train_adapter("graph")  # same recipe, Graph data

## 8. Load all three adapters onto one model
For using and testing the adapters, we load the base model **once** and attach all three adapters by name. Switching between them is instant: it just changes which small matrices are active. `disable_adapter()` switches them all off, giving us the plain base model for comparison.

In [ ]:
import json     # read and write JSON
import re       # "regular expressions": pattern matching on text
import logging  # controls how chatty libraries are

# Hide the repeated "Both max_new_tokens and max_length seem to have been set" warnings.
logging.getLogger("transformers").setLevel(logging.ERROR)

# Loading from the Prober's folder: Unsloth reads the folder, sees it is an adapter, downloads
# the base model it was trained on, and attaches the Prober adapter under the name "default".
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTERS["prober"]["save_to"],
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=hf_token,
)
# Attach the other two adapters to the same base model, each under its own name.
model.load_adapter(ADAPTERS["diagnoser"]["save_to"], adapter_name="diagnoser")
model.load_adapter(ADAPTERS["graph"]["save_to"], adapter_name="graph")

# Switch Unsloth into its faster generation mode (turns off training-only machinery).
FastLanguageModel.for_inference(model)

# Our names -> the names the adapters were registered under above.
ADAPTER_NAME = {"prober": "default", "diagnoser": "diagnoser", "graph": "graph"}


def generate(messages, adapter, max_new_tokens=256):
    """Ask the model for its next reply. adapter = "prober" / "diagnoser" / "graph", or None for the plain base model."""
    # Turn the messages into token numbers. add_generation_prompt=True adds an empty
    # "assistant" header at the end, which is how we say "now it's your turn to write".
    # return_tensors="pt" = give back a PyTorch tensor; .to("cuda") = put it on the GPU.
    ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    # do_sample=False = "greedy": always pick the single most likely next token. Same input ->
    # same output, which makes the comparison between models fair and repeatable.
    settings = dict(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    if adapter is None:
        with model.disable_adapter():  # temporarily switch every adapter off
            out = model.generate(**settings)
    else:
        model.set_adapter(ADAPTER_NAME[adapter])  # make this adapter the active one
        out = model.generate(**settings)
    # 'out' contains the prompt tokens followed by the new tokens. ids.shape[1] is the prompt
    # length, so out[0][ids.shape[1]:] keeps only the new reply. skip_special_tokens hides
    # tokens like <|eot_id|>.
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


def load_eval(name):
    """Read an adapter's held-out eval file: a list of conversations (each a list of messages)."""
    return [json.loads(line)["messages"] for line in open(f"{DATA_DIR}/{ADAPTERS[name]['eval']}")]

## 9. Quick demo: one example through each adapter
A sanity check before the full scores. Read the outputs with your own eyes: do they look like what each adapter was trained to produce?

In [ ]:
# Prober: give it the system prompt and the learner's opening message from a held-out dialogue.
prober_example = load_eval("prober")[0]
print("=== PROBER (first question for a held-out dialogue) ===")
print("Learner:", prober_example[1]["content"])
print(generate(prober_example[:2], "prober"))  # [:2] = system prompt + first learner message

# Diagnoser and Graph: give each the system prompt + the full transcript (messages[:2]),
# then compare with the correct answer stored in the file (messages[2]).
for name in ("diagnoser", "graph"):
    example = load_eval(name)[0]
    print(f"\n=== {name.upper()} ===")
    print("model  :", generate(example[:2], name, max_new_tokens=512))
    print("correct:", example[2]["content"])

## 10. Evaluate the Prober vs the base model
We replay each held-out dialogue. At every point where the assessor should speak, we give the model the real conversation so far and ask for its next reply (79 replies across 20 dialogues). We do it once with the Prober adapter and once with the plain base model. Both get the same system prompt.

Three checks per reply:
- **format-pass**: has a `<thought>` block, then exactly one question ending in `?`, and no code backticks.
- **closed-question**: the question starts with a yes/no word ("Do…", "Is…", "Can…"). Lower is better.
- **avg thought words**: how long the private reasoning is.

Last run: **fine-tuned format-pass 79/79, closed 0/79, 28 words** vs **base 68/79, 24/79, 38 words**.

In [ ]:
# A reply passes if: optional spaces, <thought>...</thought>, then text with no "<" in it.
# re.S lets "." match line breaks too, because the thought can span several lines.
def format_ok(text):
    m = re.fullmatch(r"\s*<thought>.+?</thought>\s*([^<]+)", text, re.S)
    if not m:
        return False
    question = m.group(1).strip()  # group(1) = the part after </thought>
    return question.count("?") == 1 and question.endswith("?") and "`" not in text

# Questions starting with these words can usually be answered with yes/no.
# \b = "word boundary", so "Is" matches but "Isolation" does not. re.I = ignore upper/lower case.
CLOSED = re.compile(r"^(would|do|does|did|is|are|can|could|should|will|have|has)\b", re.I)

def split_reply(text):
    """Split a reply into (thought, question). If there is no thought block, thought = None."""
    m = re.fullmatch(r"\s*<thought>(.+?)</thought>\s*(.+)", text, re.S)
    return (m.group(1).strip(), m.group(2).strip()) if m else (None, text.strip())

prober_rows = load_eval("prober")
for adapter in ("prober", None):  # None = base model
    replies = []
    for convo in prober_rows:
        for i, msg in enumerate(convo):
            if msg["role"] == "user":  # after every learner message, the assessor must reply
                replies.append(generate(convo[:i + 1], adapter))  # everything up to and including this message
    parts = [split_reply(r) for r in replies]
    thought_lengths = [len(th.split()) for th, q in parts if th]
    n = len(replies)
    print(f"{'fine-tuned' if adapter else 'base'}: "
          f"format-pass {sum(map(format_ok, replies))}/{n} | "
          f"closed-question {sum(bool(CLOSED.match(q)) for th, q in parts)}/{n} | "
          f"avg thought words {sum(thought_lengths) / max(1, len(thought_lengths)):.0f}")

## 11. Evaluate the Diagnoser vs the base model
For each of the 20 held-out transcripts, the model writes its JSON verdict. We score:
- **valid JSON**: can a program read the answer at all? An answer wrapped in Markdown code fences counts as a failure, because a program reading it directly would crash.
- **verdict type correct**: did it correctly say `none:` (learner was right), `unknown:` (not enough evidence yet) or name a real misconception? This is the most important number: it shows whether the model knows when *not* to accuse.
- **word overlap**: a rough 0–1 score of how many words the predicted misconception shares with the correct one. It is crude, since a perfect paraphrase can score low. **Read the side-by-side pairs printed below and judge them yourself.** That human judgment is the number worth quoting.

Note: the 8 eval topics never appear in training, so the model has to invent the topic name and the misconception wording.

In [ ]:
def parse_json(text):
    """Return the parsed JSON, or None if the text isn't valid JSON."""
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return None

def verdict_type(misconception):
    """'none', 'unknown', or 'real', based on the text before the first colon."""
    head = str(misconception).split(":")[0].strip().lower()
    return head if head in ("none", "unknown") else "real"

def word_overlap(a, b):
    """Share of distinct words the two texts have in common (0 = nothing, 1 = same words)."""
    wa, wb = set(re.findall(r"[a-z]+", a.lower())), set(re.findall(r"[a-z]+", b.lower()))
    return len(wa & wb) / max(1, len(wa | wb))

diag_rows = load_eval("diagnoser")
for adapter in ("diagnoser", None):
    valid = type_ok = 0
    overlaps, pairs = [], []
    for convo in diag_rows:
        gold = json.loads(convo[2]["content"])                    # the correct answer
        pred = parse_json(generate(convo[:2], adapter, max_new_tokens=128))
        if not isinstance(pred, dict) or "misconception" not in pred:
            continue                                              # unreadable answer: scores 0 on everything
        valid += 1
        type_ok += verdict_type(pred["misconception"]) == verdict_type(gold["misconception"])
        overlaps.append(word_overlap(str(pred["misconception"]), gold["misconception"]))
        pairs.append((gold["misconception"], pred["misconception"]))
    n = len(diag_rows)
    print(f"{'fine-tuned' if adapter else 'base'}: valid JSON {valid}/{n} | "
          f"verdict type correct {type_ok}/{n} | "
          f"avg word overlap {sum(overlaps) / n:.2f}")  # divide by n so unreadable answers count as 0
    if adapter:
        diag_pairs = pairs  # keep the fine-tuned pairs for reading below

print("\nCorrect vs predicted (fine-tuned). Mark each as right / partly / wrong yourself:")
for gold, pred in diag_pairs:
    print(f"- correct: {gold}\n  model  : {pred}")

## 12. Evaluate the Graph adapter vs the base model
For each held-out transcript, the model writes a concept graph. We score:
- **valid JSON**: readable by a program.
- **valid graph**: every node has a status from the allowed four, every edge points at nodes that exist, and there are no loops ("A needs B, B needs A" makes no sense for prerequisites).
- **status accuracy**: for the i-th concept in the correct graph vs the i-th concept the model wrote, do the statuses match? It is rough, because it assumes both list concepts in the same order (simple → advanced). That is how the training data is written.

We don't score exact concept names: for unseen topics the model has to invent names, and "pool_size" vs "connection_pool_size" is not really wrong.

In [ ]:
STATUSES = {"known", "shaky", "missing", "untested"}

def has_no_cycles(node_ids, edges):
    """Repeatedly remove concepts that have no remaining prerequisites. If we get stuck, there is a loop."""
    remaining = set(node_ids)
    while remaining:
        # concepts whose prerequisites are all already removed
        free = {n for n in remaining if not any(b == n and a in remaining for a, b in edges)}
        if not free:
            return False  # everything left depends on something else left: a loop
        remaining -= free
    return True

def valid_graph(g):
    try:
        ids = [node["id"] for node in g["nodes"]]
        return (len(ids) > 0 and len(set(ids)) == len(ids)                      # no duplicate concepts
                and all(node["status"] in STATUSES for node in g["nodes"])      # allowed statuses only
                and all(len(e) == 2 and e[0] in ids and e[1] in ids for e in g["edges"])  # edges point at real nodes
                and has_no_cycles(ids, g["edges"]))
    except (TypeError, KeyError):  # missing fields or wrong types = not a valid graph
        return False

graph_rows = load_eval("graph")
for adapter in ("graph", None):
    parsed = valid = matched = total = 0
    for convo in graph_rows:
        gold = json.loads(convo[2]["content"])
        pred = parse_json(generate(convo[:2], adapter, max_new_tokens=512))
        total += len(gold["nodes"])  # each gold concept is one status to get right
        if pred is None:
            continue
        parsed += 1
        if not valid_graph(pred):
            continue
        valid += 1
        # zip pairs the 1st gold concept with the 1st predicted one, 2nd with 2nd, and so on.
        matched += sum(p["status"] == g["status"] for p, g in zip(pred["nodes"], gold["nodes"]))
    n = len(graph_rows)
    print(f"{'fine-tuned' if adapter else 'base'}: valid JSON {parsed}/{n} | valid graph {valid}/{n} | "
          f"status accuracy {matched}/{total} ({matched / total:.0%})")

## 13. How to read the results
- **Fine-tuned beats base on format and validity**: the adapter learned the *output format*. Expect this to be the clearest win.
- **Verdict type and status accuracy**: these measure whether it learned the *judgement*. They are the more interesting numbers, and they are noisier with only 20 examples.
- **What these numbers are measured against**: the correct answers for the Diagnoser and Graph were written by Claude (synthetic labels). The scores show how well the model reproduces that labelling on unseen topics, not agreement with a human expert.